 # Step 1: Loading and Schema Validation

In [3]:
import polars as pl 

customers  = pl.read_csv('/home/brahimaskiou/Documents/jobsim/quantium/Data/QVI_purchase_behaviour.csv')
print(customers )

shape: (72_637, 3)
┌────────────────┬────────────────────────┬──────────────────┐
│ LYLTY_CARD_NBR ┆ LIFESTAGE              ┆ PREMIUM_CUSTOMER │
│ ---            ┆ ---                    ┆ ---              │
│ i64            ┆ str                    ┆ str              │
╞════════════════╪════════════════════════╪══════════════════╡
│ 1000           ┆ YOUNG SINGLES/COUPLES  ┆ Premium          │
│ 1002           ┆ YOUNG SINGLES/COUPLES  ┆ Mainstream       │
│ 1003           ┆ YOUNG FAMILIES         ┆ Budget           │
│ 1004           ┆ OLDER SINGLES/COUPLES  ┆ Mainstream       │
│ 1005           ┆ MIDAGE SINGLES/COUPLES ┆ Mainstream       │
│ …              ┆ …                      ┆ …                │
│ 2370651        ┆ MIDAGE SINGLES/COUPLES ┆ Mainstream       │
│ 2370701        ┆ YOUNG FAMILIES         ┆ Mainstream       │
│ 2370751        ┆ YOUNG FAMILIES         ┆ Premium          │
│ 2370961        ┆ OLDER FAMILIES         ┆ Budget           │
│ 2373711        ┆ YOUNG SINGLES/COU

In [4]:
# 2. Check the "Shape" and Schema
print("Dataset Shape:", customers.shape)
print("Schema and Types:")
print(customers.schema)

# 3. Data Quality Audit: Count Nulls across all columns
null_counts = customers.select([
    pl.all().is_null().sum()
])
print("\nNull Value Audit:")
print(null_counts)

# 4. Categorical Sanity Check: Unique Lifestages
# We sort them to easily spot duplicates with different casing
unique_lifestages = customers.select(pl.col("LIFESTAGE").unique()).sort("LIFESTAGE")
print("\nUnique Lifestages identified:")
print(unique_lifestages)

# 5. Categorical Sanity Check: Premium Status
unique_premium = customers.select(pl.col("PREMIUM_CUSTOMER").unique()).sort("PREMIUM_CUSTOMER")
print("\nUnique Premium Statuses:")
print(unique_premium)

# 6. Cardinality Check: Is LYLTY_CARD_NBR a unique Primary Key?
total_rows = customers.height
unique_cards = customers.select(pl.col("LYLTY_CARD_NBR").n_unique()).item()

print(f"\nTotal Rows: {total_rows}")
print(f"Unique Customers: {unique_cards}")

if total_rows == unique_cards:
    print("SUCCESS: Each row represents a unique customer.")
else:
    print("WARNING: Duplicate loyalty card numbers found!")

Dataset Shape: (72637, 3)
Schema and Types:
Schema({'LYLTY_CARD_NBR': Int64, 'LIFESTAGE': String, 'PREMIUM_CUSTOMER': String})

Null Value Audit:
shape: (1, 3)
┌────────────────┬───────────┬──────────────────┐
│ LYLTY_CARD_NBR ┆ LIFESTAGE ┆ PREMIUM_CUSTOMER │
│ ---            ┆ ---       ┆ ---              │
│ u32            ┆ u32       ┆ u32              │
╞════════════════╪═══════════╪══════════════════╡
│ 0              ┆ 0         ┆ 0                │
└────────────────┴───────────┴──────────────────┘

Unique Lifestages identified:
shape: (7, 1)
┌────────────────────────┐
│ LIFESTAGE              │
│ ---                    │
│ str                    │
╞════════════════════════╡
│ MIDAGE SINGLES/COUPLES │
│ NEW FAMILIES           │
│ OLDER FAMILIES         │
│ OLDER SINGLES/COUPLES  │
│ RETIREES               │
│ YOUNG FAMILIES         │
│ YOUNG SINGLES/COUPLES  │
└────────────────────────┘

Unique Premium Statuses:
shape: (3, 1)
┌──────────────────┐
│ PREMIUM_CUSTOMER │
│ ---       